In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
%cd /kaggle/working
!git clone https://github.com/ThaoTo2806/CodeFuse123.git
%cd /kaggle/working/CodeFuse123

In [ ]:
%pip install -q transformers==4.46.1 datasets==3.1.0 accelerate==1.0.1 peft==0.13.2 bitsandbytes scikit-learn pandas

In [ ]:
%pip install -U "accelerate>=1.1.0"

In [1]:
from pathlib import Path
import json
import random
import re
import time

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    set_seed,
)

DATA_DIR = "/kaggle/working/CodeFuse123/dataset"
PROMPT_VARIANT = "original"  # Run one session per value: "original" and "compact".
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MODEL_REVISION = None  # Pin the resolved commit SHA when repeating the experiment.
OUTPUT_ROOT = Path("/kaggle/working/hqcm_c1_prompt_ablation")
MAX_LENGTH = 512
VALIDATION_FRACTION = 0.10
SPLIT_SEED = 42
TRAIN_SEED = 0
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
LORA_RANK = 16
LORA_ALPHA = 32
MICRO_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
EVAL_BATCH_SIZE = 8
BOOTSTRAP_RESAMPLES = 2000
LABELS = ["style", "docs", "test", "build", "cicd", "fix", "feat", "refactor"]

if PROMPT_VARIANT not in {"original", "compact"}:
    raise ValueError("PROMPT_VARIANT must be 'original' or 'compact'.")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU before loading the quantized model.")
if DATA_DIR is None:
    candidates = sorted({
        path.parent
        for path in Path("/kaggle/input").rglob("train.json")
        if (path.parent / "test.json").is_file()
    })
    if len(candidates) != 1:
        raise RuntimeError("Set DATA_DIR to the intended folder containing train.json and test.json.")
    DATA_DIR = candidates[0]
else:
    DATA_DIR = Path(DATA_DIR)

TRAIN_PATH = DATA_DIR / "train.json"
if not TRAIN_PATH.is_file():
    raise FileNotFoundError(f"Raw train.json not found: {TRAIN_PATH}")
with TRAIN_PATH.open(encoding="utf-8") as file:
    raw_rows = json.load(file)
if len(raw_rows) != 4611:
    raise ValueError(f"Expected 4,611 official training rows; found {len(raw_rows)}.")
if any(row.get("type") not in LABELS for row in raw_rows):
    raise ValueError("The training data contains labels outside the fixed eight-label set.")

RUN_NAME = f"c1_{PROMPT_VARIANT}_seed{TRAIN_SEED}"
OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
set_seed(TRAIN_SEED)
random.seed(TRAIN_SEED)
np.random.seed(TRAIN_SEED)

ORIGINAL_TEMPLATE = """\
A git commit can typically be classified into specific categories by examining its code changes and commit message. These categories include:

- "style": Changes that solely improve the code's formatting and appearance without affecting functionality (e.g., adjusting whitespace, fixing indentation, cleaning up code formatting).
- "docs": Updates or improvements to documentation, which may include inline code comments, README files, or any other type of documentation associated with the project.
- "test": Modifications exclusively related to test code, like the addition of new tests or the correction and improvement of existing tests.
- "build": Changes that affect the build system or tools (like Gulp, Broccoli, NPM) or alterations to external dependencies (e.g., library or package updates).
- "cicd": Tweaks to configuration files or scripts used in Continuous Integration/Continuous Deployment (CI/CD) systems, such as Travis CI or CircleCI configurations.
- "fix": Code amendments that focus on rectifying errors, fixing bugs, or patching security vulnerabilities.
- "feat": Commits that introduce new features or capabilities to the project, such as new classes, functions, or methods.
- "refactor": Changes that reorganize and clean up the codebase without modifying its external behavior or outputs, improving readability and maintainability.

For a given git commit, we can inspect its code difference (diff) and the associated commit message to determine its type. Below is the diff for a specific git commit:

```
__DIFF_PLACEHOLDER__
```

Accompanying this code diff is its commit message:

```
__MESSAGE_PLACEHOLDER__
```

Given this information, the git commit can be categorized as type: """


def build_prompt(row):
    if PROMPT_VARIANT == "original":
        return ORIGINAL_TEMPLATE.replace(
            "__DIFF_PLACEHOLDER__", row["change"]
        ).replace("__MESSAGE_PLACEHOLDER__", row["summaries"]["en"])
    return (
        "Classify this commit into exactly one label: "
        "style, docs, test, build, cicd, fix, feat, refactor.\n"
        f"Diff:\n{row['change']}\n"
        f"Commit message:\n{row['summaries']['en']}\n"
        "Type:"
    )


train_indices, validation_indices = train_test_split(
    np.arange(len(raw_rows)),
    test_size=VALIDATION_FRACTION,
    random_state=SPLIT_SEED,
    stratify=[row["type"] for row in raw_rows],
)
train_rows = [raw_rows[index] for index in train_indices]
validation_rows = [raw_rows[index] for index in validation_indices]
print(f"Dataset: {DATA_DIR}")
print(f"Variant: {PROMPT_VARIANT} | train={len(train_rows)} | validation={len(validation_rows)}")

tokenizer_kwargs = {"trust_remote_code": False}
if MODEL_REVISION:
    tokenizer_kwargs["revision"] = MODEL_REVISION
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **tokenizer_kwargs)
resolved_revision = tokenizer.init_kwargs.get("_commit_hash")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def encode_example(row):
    prompt_ids = tokenizer(
        build_prompt(row),
        add_special_tokens=True,
        truncation=False,
    )["input_ids"]
    answer_ids = tokenizer(
        " " + row["type"],
        add_special_tokens=False,
    )["input_ids"]
    if tokenizer.eos_token_id is not None:
        answer_ids.append(tokenizer.eos_token_id)
    answer_ids = answer_ids[:MAX_LENGTH]
    prompt_budget = MAX_LENGTH - len(answer_ids)
    prompt_ids = prompt_ids[-prompt_budget:] if prompt_budget > 0 else []
    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


train_dataset = Dataset.from_list(train_rows)
train_dataset = train_dataset.map(
    encode_example,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing training split",
)
train_dataset = train_dataset.filter(
    lambda row: any(label != -100 for label in row["labels"]),
    desc="Dropping rows without target tokens",
)
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
    return_tensors="pt",
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=resolved_revision,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "trainer"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=MICRO_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=True,
    bf16=False,
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=1,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",
    seed=TRAIN_SEED,
    data_seed=SPLIT_SEED,
    remove_unused_columns=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)
train_started = time.perf_counter()
train_result = trainer.train()
train_seconds = time.perf_counter() - train_started
ADAPTER_OUTPUT = OUTPUT_DIR / "adapter"
trainer.save_model(str(ADAPTER_OUTPUT))
tokenizer.save_pretrained(str(ADAPTER_OUTPUT))
model.config.use_cache = True
model.eval()
input_device = model.get_input_embeddings().weight.device

label_token_ids = {
    label: tokenizer(" " + label, add_special_tokens=False)["input_ids"]
    + ([tokenizer.eos_token_id] if tokenizer.eos_token_id is not None else [])
    for label in LABELS
}


def score_prompt(prompt):
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )["input_ids"]
    sequences = [prompt_ids + label_token_ids[label] for label in LABELS]
    padded_length = max(len(sequence) for sequence in sequences)
    input_ids = torch.full(
        (len(sequences), padded_length),
        tokenizer.pad_token_id,
        dtype=torch.long,
    )
    attention_mask = torch.zeros_like(input_ids)
    for row_index, sequence in enumerate(sequences):
        input_ids[row_index, :len(sequence)] = torch.tensor(sequence)
        attention_mask[row_index, :len(sequence)] = 1
    input_ids = input_ids.to(input_device)
    attention_mask = attention_mask.to(input_device)

    with torch.inference_mode():
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        ).logits.float()
        log_probabilities = torch.log_softmax(logits, dim=-1)

    scores = []
    prompt_length = len(prompt_ids)
    for row_index, label in enumerate(LABELS):
        target_ids = label_token_ids[label]
        positions = torch.arange(
            prompt_length - 1,
            prompt_length + len(target_ids) - 1,
            device=input_device,
        )
        targets = torch.tensor(target_ids, dtype=torch.long, device=input_device)
        scores.append(log_probabilities[row_index, positions, targets].sum())
    return torch.stack(scores).cpu()


gold_labels = [row["type"] for row in validation_rows]
predictions = []
probabilities = []
prompt_lengths = []
for index, row in enumerate(validation_rows, start=1):
    prompt = build_prompt(row)
    prompt_lengths.append(len(tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=False,
    )["input_ids"]))
    scores = score_prompt(prompt)
    class_probabilities = torch.softmax(scores, dim=0).numpy()
    predictions.append(LABELS[int(class_probabilities.argmax())])
    probabilities.append(class_probabilities)
    if index % 50 == 0 or index == len(validation_rows):
        print(f"Scored validation {index}/{len(validation_rows)}")

probabilities = np.asarray(probabilities, dtype=np.float64)
confidence = probabilities.max(axis=1)
correct = np.asarray(predictions) == np.asarray(gold_labels)
bin_edges = np.linspace(0.0, 1.0, 11)
ece = 0.0
for bin_index in range(10):
    lower, upper = bin_edges[bin_index], bin_edges[bin_index + 1]
    in_bin = (
        (confidence >= lower) & (confidence <= upper)
        if bin_index == 9
        else (confidence >= lower) & (confidence < upper)
    )
    if in_bin.any():
        ece += in_bin.mean() * abs(correct[in_bin].mean() - confidence[in_bin].mean())

metrics = {
    "prompt_variant": PROMPT_VARIANT,
    "model_id": MODEL_ID,
    "model_revision": resolved_revision,
    "seed": TRAIN_SEED,
    "split_seed": SPLIT_SEED,
    "train_samples": len(train_rows),
    "validation_samples": len(validation_rows),
    "max_length": MAX_LENGTH,
    "mean_prompt_tokens_before_truncation": float(np.mean(prompt_lengths)),
    "median_prompt_tokens_before_truncation": float(np.median(prompt_lengths)),
    "train_seconds": train_seconds,
    "train_loss": float(train_result.training_loss),
    "accuracy": float(accuracy_score(gold_labels, predictions)),
    "macro_f1_8": float(f1_score(
        gold_labels, predictions, labels=LABELS, average="macro", zero_division=0
    )),
    "ece_10_bins_raw": float(ece),
    "ece_note": "Exploratory, uncalibrated softmax over eight label log-likelihoods.",
    "invalid_count": 0,
    "inference": "Constrained to the fixed eight-label set using summed label-plus-EOS log-likelihood.",
}
print(classification_report(
    gold_labels,
    predictions,
    labels=LABELS,
    target_names=LABELS,
    zero_division=0,
    digits=4,
))
print(json.dumps(metrics, ensure_ascii=False, indent=2))
pd.DataFrame({
    "sample_id": [f"train:{index}" for index in validation_indices],
    "gold_label": gold_labels,
    "predicted_label": predictions,
    "confidence": confidence,
}).to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)
with (OUTPUT_DIR / "metrics.json").open("w", encoding="utf-8") as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)
with (OUTPUT_DIR / "split_indices.json").open("w", encoding="utf-8") as file:
    json.dump({
        "split_seed": SPLIT_SEED,
        "train_indices": [int(index) for index in train_indices],
        "validation_indices": [int(index) for index in validation_indices],
    }, file, ensure_ascii=False)
print("Saved C1 artifacts to:", OUTPUT_DIR)

Dataset: /kaggle/working/CodeFuse123/dataset
Variant: original | train=4149 | validation=462


Tokenizing training split:   0%|          | 0/4149 [00:00<?, ? examples/s]

Dropping rows without target tokens:   0%|          | 0/4149 [00:00<?, ? examples/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as subsequent forwards. If the mismatch is intentional, you can use torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False) to suppress this warning. (Triggered internally at /pytorch/torch/csrc/autograd/input_buffer.cpp:240.)
  return Variable._execution_en

Step,Training Loss
25,4.956300
50,0.800500
75,0.535800
100,0.407900
125,0.495300
150,0.366100
175,0.461000
200,0.403100
225,0.451600
250,0.501800


Scored validation 50/462
Scored validation 100/462
Scored validation 150/462
Scored validation 200/462
Scored validation 250/462
Scored validation 300/462
Scored validation 350/462
Scored validation 400/462
Scored validation 450/462
Scored validation 462/462
              precision    recall  f1-score   support

       style     0.2727    0.3333    0.3000         9
        docs     0.1818    0.1667    0.1739        36
        test     0.7917    0.4222    0.5507        45
       build     0.0833    0.1111    0.0952         9
        cicd     0.0659    0.6667    0.1200         9
         fix     0.4688    0.1261    0.1987       119
        feat     0.3556    0.1569    0.2177       102
    refactor     0.3505    0.5639    0.4323       133

    accuracy                         0.3052       462
   macro avg     0.3213    0.3184    0.2611       462
weighted avg     0.3996    0.3052    0.3009       462

{
  "prompt_variant": "original",
  "model_id": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
  "mod

In [2]:
import shutil

shutil.make_archive(
    "/kaggle/working/hqcm_c1_prompt_ablation",
    "zip",
    "/kaggle/working",
    "hqcm_c1_prompt_ablation",
)

print("ZIP saved to: /kaggle/working/hqcm_c1_prompt_ablation.zip")

ZIP saved to: /kaggle/working/hqcm_c1_prompt_ablation.zip


In [3]:
from pathlib import Path
import json
import random
import re
import time

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    set_seed,
)

DATA_DIR = "/kaggle/working/CodeFuse123/dataset"
PROMPT_VARIANT = "compact"  # Run one session per value: "original" and "compact".
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MODEL_REVISION = None  # Pin the resolved commit SHA when repeating the experiment.
OUTPUT_ROOT = Path("/kaggle/working/hqcm_c1_prompt_compact_ablation")
MAX_LENGTH = 512
VALIDATION_FRACTION = 0.10
SPLIT_SEED = 42
TRAIN_SEED = 0
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
LORA_RANK = 16
LORA_ALPHA = 32
MICRO_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
EVAL_BATCH_SIZE = 8
BOOTSTRAP_RESAMPLES = 2000
LABELS = ["style", "docs", "test", "build", "cicd", "fix", "feat", "refactor"]

if PROMPT_VARIANT not in {"original", "compact"}:
    raise ValueError("PROMPT_VARIANT must be 'original' or 'compact'.")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU before loading the quantized model.")
if DATA_DIR is None:
    candidates = sorted({
        path.parent
        for path in Path("/kaggle/input").rglob("train.json")
        if (path.parent / "test.json").is_file()
    })
    if len(candidates) != 1:
        raise RuntimeError("Set DATA_DIR to the intended folder containing train.json and test.json.")
    DATA_DIR = candidates[0]
else:
    DATA_DIR = Path(DATA_DIR)

TRAIN_PATH = DATA_DIR / "train.json"
if not TRAIN_PATH.is_file():
    raise FileNotFoundError(f"Raw train.json not found: {TRAIN_PATH}")
with TRAIN_PATH.open(encoding="utf-8") as file:
    raw_rows = json.load(file)
if len(raw_rows) != 4611:
    raise ValueError(f"Expected 4,611 official training rows; found {len(raw_rows)}.")
if any(row.get("type") not in LABELS for row in raw_rows):
    raise ValueError("The training data contains labels outside the fixed eight-label set.")

RUN_NAME = f"c1_{PROMPT_VARIANT}_seed{TRAIN_SEED}"
OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
set_seed(TRAIN_SEED)
random.seed(TRAIN_SEED)
np.random.seed(TRAIN_SEED)

ORIGINAL_TEMPLATE = """\
A git commit can typically be classified into specific categories by examining its code changes and commit message. These categories include:

- "style": Changes that solely improve the code's formatting and appearance without affecting functionality (e.g., adjusting whitespace, fixing indentation, cleaning up code formatting).
- "docs": Updates or improvements to documentation, which may include inline code comments, README files, or any other type of documentation associated with the project.
- "test": Modifications exclusively related to test code, like the addition of new tests or the correction and improvement of existing tests.
- "build": Changes that affect the build system or tools (like Gulp, Broccoli, NPM) or alterations to external dependencies (e.g., library or package updates).
- "cicd": Tweaks to configuration files or scripts used in Continuous Integration/Continuous Deployment (CI/CD) systems, such as Travis CI or CircleCI configurations.
- "fix": Code amendments that focus on rectifying errors, fixing bugs, or patching security vulnerabilities.
- "feat": Commits that introduce new features or capabilities to the project, such as new classes, functions, or methods.
- "refactor": Changes that reorganize and clean up the codebase without modifying its external behavior or outputs, improving readability and maintainability.

For a given git commit, we can inspect its code difference (diff) and the associated commit message to determine its type. Below is the diff for a specific git commit:

```
__DIFF_PLACEHOLDER__
```

Accompanying this code diff is its commit message:

```
__MESSAGE_PLACEHOLDER__
```

Given this information, the git commit can be categorized as type: """


def build_prompt(row):
    if PROMPT_VARIANT == "original":
        return ORIGINAL_TEMPLATE.replace(
            "__DIFF_PLACEHOLDER__", row["change"]
        ).replace("__MESSAGE_PLACEHOLDER__", row["summaries"]["en"])
    return (
        "Classify this commit into exactly one label: "
        "style, docs, test, build, cicd, fix, feat, refactor.\n"
        f"Diff:\n{row['change']}\n"
        f"Commit message:\n{row['summaries']['en']}\n"
        "Type:"
    )


train_indices, validation_indices = train_test_split(
    np.arange(len(raw_rows)),
    test_size=VALIDATION_FRACTION,
    random_state=SPLIT_SEED,
    stratify=[row["type"] for row in raw_rows],
)
train_rows = [raw_rows[index] for index in train_indices]
validation_rows = [raw_rows[index] for index in validation_indices]
print(f"Dataset: {DATA_DIR}")
print(f"Variant: {PROMPT_VARIANT} | train={len(train_rows)} | validation={len(validation_rows)}")

tokenizer_kwargs = {"trust_remote_code": False}
if MODEL_REVISION:
    tokenizer_kwargs["revision"] = MODEL_REVISION
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **tokenizer_kwargs)
resolved_revision = tokenizer.init_kwargs.get("_commit_hash")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def encode_example(row):
    prompt_ids = tokenizer(
        build_prompt(row),
        add_special_tokens=True,
        truncation=False,
    )["input_ids"]
    answer_ids = tokenizer(
        " " + row["type"],
        add_special_tokens=False,
    )["input_ids"]
    if tokenizer.eos_token_id is not None:
        answer_ids.append(tokenizer.eos_token_id)
    answer_ids = answer_ids[:MAX_LENGTH]
    prompt_budget = MAX_LENGTH - len(answer_ids)
    prompt_ids = prompt_ids[-prompt_budget:] if prompt_budget > 0 else []
    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


train_dataset = Dataset.from_list(train_rows)
train_dataset = train_dataset.map(
    encode_example,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing training split",
)
train_dataset = train_dataset.filter(
    lambda row: any(label != -100 for label in row["labels"]),
    desc="Dropping rows without target tokens",
)
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
    return_tensors="pt",
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=resolved_revision,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "trainer"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=MICRO_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=True,
    bf16=False,
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=1,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",
    seed=TRAIN_SEED,
    data_seed=SPLIT_SEED,
    remove_unused_columns=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)
train_started = time.perf_counter()
train_result = trainer.train()
train_seconds = time.perf_counter() - train_started
ADAPTER_OUTPUT = OUTPUT_DIR / "adapter"
trainer.save_model(str(ADAPTER_OUTPUT))
tokenizer.save_pretrained(str(ADAPTER_OUTPUT))
model.config.use_cache = True
model.eval()
input_device = model.get_input_embeddings().weight.device

label_token_ids = {
    label: tokenizer(" " + label, add_special_tokens=False)["input_ids"]
    + ([tokenizer.eos_token_id] if tokenizer.eos_token_id is not None else [])
    for label in LABELS
}


def score_prompt(prompt):
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )["input_ids"]
    sequences = [prompt_ids + label_token_ids[label] for label in LABELS]
    padded_length = max(len(sequence) for sequence in sequences)
    input_ids = torch.full(
        (len(sequences), padded_length),
        tokenizer.pad_token_id,
        dtype=torch.long,
    )
    attention_mask = torch.zeros_like(input_ids)
    for row_index, sequence in enumerate(sequences):
        input_ids[row_index, :len(sequence)] = torch.tensor(sequence)
        attention_mask[row_index, :len(sequence)] = 1
    input_ids = input_ids.to(input_device)
    attention_mask = attention_mask.to(input_device)

    with torch.inference_mode():
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        ).logits.float()
        log_probabilities = torch.log_softmax(logits, dim=-1)

    scores = []
    prompt_length = len(prompt_ids)
    for row_index, label in enumerate(LABELS):
        target_ids = label_token_ids[label]
        positions = torch.arange(
            prompt_length - 1,
            prompt_length + len(target_ids) - 1,
            device=input_device,
        )
        targets = torch.tensor(target_ids, dtype=torch.long, device=input_device)
        scores.append(log_probabilities[row_index, positions, targets].sum())
    return torch.stack(scores).cpu()


gold_labels = [row["type"] for row in validation_rows]
predictions = []
probabilities = []
prompt_lengths = []
for index, row in enumerate(validation_rows, start=1):
    prompt = build_prompt(row)
    prompt_lengths.append(len(tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=False,
    )["input_ids"]))
    scores = score_prompt(prompt)
    class_probabilities = torch.softmax(scores, dim=0).numpy()
    predictions.append(LABELS[int(class_probabilities.argmax())])
    probabilities.append(class_probabilities)
    if index % 50 == 0 or index == len(validation_rows):
        print(f"Scored validation {index}/{len(validation_rows)}")

probabilities = np.asarray(probabilities, dtype=np.float64)
confidence = probabilities.max(axis=1)
correct = np.asarray(predictions) == np.asarray(gold_labels)
bin_edges = np.linspace(0.0, 1.0, 11)
ece = 0.0
for bin_index in range(10):
    lower, upper = bin_edges[bin_index], bin_edges[bin_index + 1]
    in_bin = (
        (confidence >= lower) & (confidence <= upper)
        if bin_index == 9
        else (confidence >= lower) & (confidence < upper)
    )
    if in_bin.any():
        ece += in_bin.mean() * abs(correct[in_bin].mean() - confidence[in_bin].mean())

metrics = {
    "prompt_variant": PROMPT_VARIANT,
    "model_id": MODEL_ID,
    "model_revision": resolved_revision,
    "seed": TRAIN_SEED,
    "split_seed": SPLIT_SEED,
    "train_samples": len(train_rows),
    "validation_samples": len(validation_rows),
    "max_length": MAX_LENGTH,
    "mean_prompt_tokens_before_truncation": float(np.mean(prompt_lengths)),
    "median_prompt_tokens_before_truncation": float(np.median(prompt_lengths)),
    "train_seconds": train_seconds,
    "train_loss": float(train_result.training_loss),
    "accuracy": float(accuracy_score(gold_labels, predictions)),
    "macro_f1_8": float(f1_score(
        gold_labels, predictions, labels=LABELS, average="macro", zero_division=0
    )),
    "ece_10_bins_raw": float(ece),
    "ece_note": "Exploratory, uncalibrated softmax over eight label log-likelihoods.",
    "invalid_count": 0,
    "inference": "Constrained to the fixed eight-label set using summed label-plus-EOS log-likelihood.",
}
print(classification_report(
    gold_labels,
    predictions,
    labels=LABELS,
    target_names=LABELS,
    zero_division=0,
    digits=4,
))
print(json.dumps(metrics, ensure_ascii=False, indent=2))
pd.DataFrame({
    "sample_id": [f"train:{index}" for index in validation_indices],
    "gold_label": gold_labels,
    "predicted_label": predictions,
    "confidence": confidence,
}).to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)
with (OUTPUT_DIR / "metrics.json").open("w", encoding="utf-8") as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)
with (OUTPUT_DIR / "split_indices.json").open("w", encoding="utf-8") as file:
    json.dump({
        "split_seed": SPLIT_SEED,
        "train_indices": [int(index) for index in train_indices],
        "validation_indices": [int(index) for index in validation_indices],
    }, file, ensure_ascii=False)
print("Saved C1 artifacts to:", OUTPUT_DIR)

Dataset: /kaggle/working/CodeFuse123/dataset
Variant: compact | train=4149 | validation=462


Tokenizing training split:   0%|          | 0/4149 [00:00<?, ? examples/s]

Dropping rows without target tokens:   0%|          | 0/4149 [00:00<?, ? examples/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Step,Training Loss
25,3.124500
50,0.730900
75,0.552700
100,0.441100
125,0.565500
150,0.446400
175,0.419400
200,0.366600
225,0.448000
250,0.504400


Scored validation 50/462
Scored validation 100/462
Scored validation 150/462
Scored validation 200/462
Scored validation 250/462
Scored validation 300/462
Scored validation 350/462
Scored validation 400/462
Scored validation 450/462
Scored validation 462/462
              precision    recall  f1-score   support

       style     1.0000    0.8889    0.9412         9
        docs     0.9286    0.7222    0.8125        36
        test     0.9111    0.9111    0.9111        45
       build     0.9000    1.0000    0.9474         9
        cicd     0.9000    1.0000    0.9474         9
         fix     0.6812    0.7899    0.7315       119
        feat     0.8191    0.7549    0.7857       102
    refactor     0.7829    0.7594    0.7710       133

    accuracy                         0.7900       462
   macro avg     0.8654    0.8533    0.8560       462
weighted avg     0.7973    0.7900    0.7911       462

{
  "prompt_variant": "compact",
  "model_id": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
  "mode

In [4]:
import shutil

shutil.make_archive(
    "/kaggle/working/hqcm_c1_prompt_compact_ablation",
    "zip",
    "/kaggle/working",
    "hqcm_c1_prompt_compact_ablation",
)

print("ZIP saved to: /kaggle/working/hqcm_c1_prompt_compact_ablation.zip")

ZIP saved to: /kaggle/working/hqcm_c1_prompt_compact_ablation.zip


In [ ]:
%pip install -q transformers==4.46.1 peft==0.13.2 bitsandbytes scikit-learn pandas

In [ ]:
%pip install -U peft

In [ ]:
from huggingface_hub import HfApi
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
import torch
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

DATA_DIR = "/kaggle/working/CodeFuse123/dataset"  # Example: Path("/kaggle/input/hqcm/dataset")
ADAPTER_DIR = "/kaggle/input/datasets/phpadmin123/adapter-run1/hqcm_llama2_7b_chcl_fixed_v2"  # Example: Path("/kaggle/input/hqcm-run1/adapter")
OUTPUT_DIR = Path("/kaggle/working/hqcm_huong_b_c3")
MODEL_ID = "meta-llama/Llama-2-7b-hf"
MODEL_REVISION = None  # Set a pinned Hugging Face commit SHA for repeat runs.
MAX_INPUT_TOKENS = 512
BATCH_SIZE = 8
BOOTSTRAP_RESAMPLES = 2000
ECE_BINS = 10
SEED = 42
LABELS = ["style", "docs", "test", "build", "cicd", "fix", "feat", "refactor"]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if DATA_DIR is None:
    data_dirs = sorted({
        path.parent
        for path in Path("/kaggle/input").rglob("train.json")
        if (path.parent / "test.json").is_file()
    })
    if len(data_dirs) != 1:
        raise RuntimeError(
            "Set DATA_DIR to the folder containing the intended raw train.json and test.json."
        )
    DATA_DIR = data_dirs[0]
else:
    DATA_DIR = Path(DATA_DIR)

if ADAPTER_DIR is None:
    raise RuntimeError(
        "Set ADAPTER_DIR to the uploaded Run 1 adapter. "
        "If it is unavailable, retrain the verified Run 1 adapter first; do not use Run 2."
    )
ADAPTER_DIR = Path(ADAPTER_DIR)
if not (ADAPTER_DIR / "adapter_config.json").is_file():
    raise FileNotFoundError(f"Adapter config not found under {ADAPTER_DIR}")
if not (ADAPTER_DIR / "adapter_model.safetensors").is_file():
    raise FileNotFoundError(f"Adapter weights not found under {ADAPTER_DIR}")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU before loading the 7B model.")

train_path = DATA_DIR / "train.json"
test_path = DATA_DIR / "test.json"
if not train_path.is_file() or not test_path.is_file():
    raise FileNotFoundError(f"Expected raw train.json and test.json under {DATA_DIR}")

with train_path.open(encoding="utf-8") as file:
    train_rows = json.load(file)
with test_path.open(encoding="utf-8") as file:
    test_rows = json.load(file)
if len(train_rows) != 4611 or len(test_rows) != 518:
    raise ValueError(
        f"Expected official split 4611/518; got {len(train_rows)}/{len(test_rows)}."
    )

CHCL_PROMPT_TEMPLATE = """\
A git commit can typically be classified into specific categories by examining its code changes and commit message. These categories include:

- "style": Changes that solely improve the code's formatting and appearance without affecting functionality (e.g., adjusting whitespace, fixing indentation, cleaning up code formatting).
- "docs": Updates or improvements to documentation, which may include inline code comments, README files, or any other type of documentation associated with the project.
- "test": Modifications exclusively related to test code, like the addition of new tests or the correction and improvement of existing tests.
- "build": Changes that affect the build system or tools (like Gulp, Broccoli, NPM) or alterations to external dependencies (e.g., library or package updates).
- "cicd": Tweaks to configuration files or scripts used in Continuous Integration/Continuous Deployment (CI/CD) systems, such as Travis CI or CircleCI configurations.
- "fix": Code amendments that focus on rectifying errors, fixing bugs, or patching security vulnerabilities.
- "feat": Commits that introduce new features or capabilities to the project, such as new classes, functions, or methods.
- "refactor": Changes that reorganize and clean up the codebase without modifying its external behavior or outputs, improving readability and maintainability.

For a given git commit, we can inspect its code difference (diff) and the associated commit message to determine its type. Below is the diff for a specific git commit:

```

{diff}

```

Accompanying this code diff is its commit message:

```

{message}

```

Given this information, the git commit can be categorized as type: """

if any(row.get("type") not in LABELS for row in train_rows + test_rows):
    raise ValueError("The raw dataset contains labels outside the fixed eight-label set.")

hf_token = os.environ.get("kaggle-llama")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient

        hf_token = UserSecretsClient().get_secret("kaggle-llama")
    except Exception as error:
        raise RuntimeError("Add the Hugging Face token as Kaggle Secret HF_TOKEN.") from error

tokenizer_kwargs = {"token": hf_token}
# Resolve the exact base-model commit from Hugging Face Hub.
if MODEL_REVISION is None:
    model_info = HfApi().model_info(
        repo_id=MODEL_ID,
        token=hf_token,
        revision="main",
    )
    resolved_revision = model_info.sha
else:
    resolved_revision = MODEL_REVISION

if not resolved_revision:
    raise RuntimeError(
        "Could not resolve the base-model commit SHA. "
        "Check Hugging Face access and MODEL_REVISION."
    )

print("Base model:", MODEL_ID)
print("Resolved commit:", resolved_revision)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=hf_token,
    revision=resolved_revision,
)

tokenizer.truncation_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

if tokenizer.eos_token_id is None:
    raise ValueError(
        "The tokenizer must define an EOS token for candidate scoring."
    )

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    revision=resolved_revision,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model.eval()
input_device = model.get_input_embeddings().weight.device


def build_prompt(row):
    return CHCL_PROMPT_TEMPLATE.format(
        diff=row["change"],
        message=row["summaries"]["en"],
    )


def candidate_token_ids(label):
    label_ids = tokenizer(" " + label, add_special_tokens=False)["input_ids"]
    if not label_ids:
        raise ValueError(f"Empty tokenization for label: {label}")
    return label_ids + [tokenizer.eos_token_id]


label_token_ids = {label: candidate_token_ids(label) for label in LABELS}


def score_candidates(prompt):
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )["input_ids"]
    sequences = [
        prompt_ids + label_token_ids[label]
        for label in LABELS
    ]
    padded_length = max(len(sequence) for sequence in sequences)
    input_ids = torch.full(
        (len(sequences), padded_length),
        fill_value=tokenizer.pad_token_id,
        dtype=torch.long,
    )
    attention_mask = torch.zeros_like(input_ids)
    for row_index, sequence in enumerate(sequences):
        input_ids[row_index, :len(sequence)] = torch.tensor(sequence, dtype=torch.long)
        attention_mask[row_index, :len(sequence)] = 1

    input_ids = input_ids.to(input_device)
    attention_mask = attention_mask.to(input_device)
    with torch.inference_mode():
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        ).logits.float()
        log_probabilities = torch.log_softmax(logits, dim=-1)

    scores = []
    prompt_length = len(prompt_ids)
    for row_index, label in enumerate(LABELS):
        target_ids = label_token_ids[label]
        target_positions = torch.arange(
            prompt_length - 1,
            prompt_length + len(target_ids) - 1,
            device=input_device,
        )
        target_tokens = torch.tensor(target_ids, dtype=torch.long, device=input_device)
        token_scores = log_probabilities[row_index, target_positions, target_tokens]
        scores.append(token_scores.sum())

    return torch.stack(scores).cpu()


gold_labels = [str(row["type"]).strip().lower() for row in test_rows]
score_rows = []
predictions = []
probabilities = []

for index, row in enumerate(test_rows, start=1):
    scores = score_candidates(build_prompt(row))
    class_probabilities = torch.softmax(scores, dim=0).numpy()
    prediction = LABELS[int(class_probabilities.argmax())]
    predictions.append(prediction)
    probabilities.append(class_probabilities)
    score_rows.append({
        "index": index - 1,
        "gold_label": gold_labels[index - 1],
        "predicted_label": prediction,
        "confidence": float(class_probabilities.max()),
        **{f"prob_{label}": float(class_probabilities[label_index])
           for label_index, label in enumerate(LABELS)},
    })
    if index % 50 == 0 or index == len(test_rows):
        print(f"Scored {index}/{len(test_rows)}")

assert len(predictions) == len(test_rows)
assert set(predictions).issubset(set(LABELS))
probability_array = np.asarray(probabilities, dtype=np.float64)
confidence = probability_array.max(axis=1)
correct = np.asarray(predictions) == np.asarray(gold_labels)


def expected_calibration_error(confidence_values, correct_values, n_bins=ECE_BINS):
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for bin_index in range(n_bins):
        lower = bin_edges[bin_index]
        upper = bin_edges[bin_index + 1]
        if bin_index == n_bins - 1:
            in_bin = (confidence_values >= lower) & (confidence_values <= upper)
        else:
            in_bin = (confidence_values >= lower) & (confidence_values < upper)
        if in_bin.any():
            ece += in_bin.mean() * abs(
                correct_values[in_bin].mean() - confidence_values[in_bin].mean()
            )
    return float(ece)


def calculate_metrics(indices):
    selected_gold = np.asarray(gold_labels)[indices]
    selected_predictions = np.asarray(predictions)[indices]
    return {
        "accuracy": float(accuracy_score(selected_gold, selected_predictions)),
        "macro_f1_8": float(f1_score(
            selected_gold,
            selected_predictions,
            labels=LABELS,
            average="macro",
            zero_division=0,
        )),
    }


all_indices = np.arange(len(test_rows))
point_metrics = calculate_metrics(all_indices)
rng = np.random.default_rng(SEED)
bootstrap_values = {name: [] for name in point_metrics}
for _ in range(BOOTSTRAP_RESAMPLES):
    sample_indices = rng.integers(0, len(test_rows), size=len(test_rows))
    sample_metrics = calculate_metrics(sample_indices)
    for metric_name, metric_value in sample_metrics.items():
        bootstrap_values[metric_name].append(metric_value)

metrics = {
    **point_metrics,
    "macro_precision_8": float(precision_score(
        gold_labels, predictions, labels=LABELS, average="macro", zero_division=0
    )),
    "macro_recall_8": float(recall_score(
        gold_labels, predictions, labels=LABELS, average="macro", zero_division=0
    )),
    "micro_f1_8": float(f1_score(
        gold_labels, predictions, labels=LABELS, average="micro", zero_division=0
    )),
    "invalid_count": 0,
    "invalid_rate": 0.0,
    "ece_10_bins_raw": expected_calibration_error(confidence, correct),
    "ece_note": "Uncalibrated ECE of softmax over summed label-plus-EOS log-likelihoods; exploratory only.",
    "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
    "bootstrap_seed": SEED,
    "n_samples": len(test_rows),
    "labels": LABELS,
    "base_model": MODEL_ID,
    "base_model_revision": resolved_revision,
    "adapter_dir": str(ADAPTER_DIR),
    "max_input_tokens_left_truncated": MAX_INPUT_TOKENS,
    "scoring": "summed conditional token log-likelihood of label plus EOS",
}
for metric_name, values in bootstrap_values.items():
    metrics[f"{metric_name}_bootstrap_95ci"] = [
        float(np.quantile(values, 0.025)),
        float(np.quantile(values, 0.975)),
    ]

print(classification_report(
    gold_labels,
    predictions,
    labels=LABELS,
    target_names=LABELS,
    zero_division=0,
    digits=4,
))
print(json.dumps(metrics, ensure_ascii=False, indent=2))
pd.DataFrame(score_rows).to_csv(OUTPUT_DIR / "c3_test_scores.csv", index=False)
with (OUTPUT_DIR / "c3_metrics.json").open("w", encoding="utf-8") as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)
print("Saved C3 artifacts to:", OUTPUT_DIR)

In [ ]:
!pip install -q transformers==4.46.1 pandas

In [ ]:
from pathlib import Path
import json
import random
import re

import pandas as pd

DATA_DIR = None  # # Example: Path("/kaggle/input/hqcm/dataset")
OUTPUT_DIR = Path("/kaggle/working/improve_step2_manual_labeling")
RANDOM_SEED = 42
UNSUPPORTED_IDENTIFIER_SAMPLES = 50
LOW_COVERAGE_SAMPLES = 50
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IDENTIFIER_PATTERN = re.compile(
    r"(?<![A-Za-z0-9])(?:"
    r"[A-Za-z][A-Za-z0-9]*_[A-Za-z0-9_]*|"
    r"[A-Za-z]+[A-Z][A-Za-z0-9]*|"
    r"[A-Z]{2,}[A-Za-z0-9]*"
    r")(?![A-Za-z0-9])"
)
WORD_PATTERN = re.compile(r"[A-Za-z][A-Za-z0-9_]*")
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "being",
    "by", "for", "from", "in", "into", "is", "it", "its", "of",
    "on", "or", "that", "the", "their", "them", "then", "there",
    "these", "this", "those", "to", "was", "were", "will", "with",
}
LABEL_VALUES = [
    "hallucination",
    "valid concept",
    "context outside diff",
    "other",
]


def load_raw_split(train_path, test_path):
    try:
        train_rows = json.loads(train_path.read_text(encoding="utf-8"))
        test_rows = json.loads(test_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None

    if not isinstance(train_rows, list) or not isinstance(test_rows, list):
        return None
    if not train_rows or not test_rows:
        return None

    required_keys = {"change", "summaries", "type"}
    for rows in (train_rows, test_rows):
        first_row = rows[0]
        if not isinstance(first_row, dict) or not required_keys.issubset(first_row):
            return None
        if not isinstance(first_row.get("summaries"), dict):
            return None
        if "en" not in first_row["summaries"]:
            return None
    return train_rows, test_rows


if DATA_DIR is not None:
    candidate_dirs = [Path(DATA_DIR)]
else:
    search_roots = [Path("/kaggle/input"), Path("/kaggle/working")]
    candidate_dirs = sorted({
        path.parent
        for root in search_roots
        if root.exists()
        for path in root.rglob("train.json")
    })

valid_datasets = []
for directory in candidate_dirs:
    train_path = directory / "train.json"
    test_path = directory / "test.json"
    if not train_path.is_file() or not test_path.is_file():
        continue
    loaded = load_raw_split(train_path, test_path)
    if loaded is not None:
        valid_datasets.append((directory, *loaded))

if not valid_datasets:
    raise FileNotFoundError(
    "Could not find the original HQCM dataset containing "
    "train.json/test.json and the required fields "
    "change, summaries.en, and type. Please set DATA_DIR."
)

if len(valid_datasets) > 1:
    official_size = [
        item for item in valid_datasets
        if len(item[1]) + len(item[2]) == 5129
    ]
    if len(official_size) == 1:
        valid_datasets = official_size
    else:
        found_dirs = [str(item[0]) for item in valid_datasets]
        raise RuntimeError(
    "Multiple HQCM datasets were found. Please set DATA_DIR "
    "to the intended dataset directory: "
    + ", ".join(found_dirs)
)

dataset_dir, train_rows, test_rows = valid_datasets[0]
print(f"Dataset: {dataset_dir}")
print(f"Train: {len(train_rows):,} | Test: {len(test_rows):,}")


def unique_identifiers(text):
    found = IDENTIFIER_PATTERN.findall(str(text))
    return list(dict.fromkeys(found))


def content_terms(text):
    return [
        word.lower()
        for word in WORD_PATTERN.findall(str(text))
        if len(word) > 1 and word.lower() not in STOPWORDS
    ]


records = []
for split_name, rows in (("train", train_rows), ("test", test_rows)):
    for row_index, row in enumerate(rows):
        summary = str(row["summaries"].get("en", "")).strip()
        diff = str(row["change"])
        if not summary:
            continue

        diff_lower = diff.lower()
        summary_identifiers = unique_identifiers(summary)
        unsupported_identifiers = [
            identifier
            for identifier in summary_identifiers
            if identifier.lower() not in diff_lower
        ]

        summary_terms = content_terms(summary)
        diff_terms = {word.lower() for word in WORD_PATTERN.findall(diff)}
        coverage = (
            sum(term in diff_terms for term in summary_terms) / len(summary_terms)
            if summary_terms else 0.0
        )
        terms_not_found = list(dict.fromkeys(
            term for term in summary_terms if term not in diff_terms
        ))

        records.append({
            "sample_id": f"{split_name}:{row_index}",
            "split": split_name,
            "row_index": row_index,
            "type": row.get("type", ""),
            "subtype": row.get("subtype", ""),
            "summary_en": summary,
            "diff": diff,
            "unmatched_identifiers": unsupported_identifiers,
            "lexical_coverage": coverage,
            "summary_terms_not_found_in_diff": terms_not_found,
        })

unsupported_pool = [record for record in records if record["unmatched_identifiers"]]
if len(unsupported_pool) < UNSUPPORTED_IDENTIFIER_SAMPLES:
    raise ValueError(
    f"Only {len(unsupported_pool)} samples have flagged identifiers; "
    f"{UNSUPPORTED_IDENTIFIER_SAMPLES} samples are required."
)

random_generator = random.Random(RANDOM_SEED)
unsupported_selected = random_generator.sample(
    unsupported_pool,
    UNSUPPORTED_IDENTIFIER_SAMPLES,
)
selected_ids = {record["sample_id"] for record in unsupported_selected}

# The low-coverage group excludes samples with flagged identifiers
# to keep the two sampling groups distinct.
low_coverage_pool = [
    record for record in records
    if not record["unmatched_identifiers"]
    and record["sample_id"] not in selected_ids
]
if len(low_coverage_pool) < LOW_COVERAGE_SAMPLES:
    raise ValueError(
    f"Only {len(low_coverage_pool)} samples without flagged identifiers "
    f"are available; {LOW_COVERAGE_SAMPLES} samples are required "
    "for the low-coverage group."
)

low_coverage_selected = sorted(
    low_coverage_pool,
    key=lambda record: (
        record["lexical_coverage"],
        record["split"],
        record["row_index"],
    ),
)[:LOW_COVERAGE_SAMPLES]

annotation_rows = []
for group_name, selected in (
    ("unsupported_identifier", unsupported_selected),
    ("low_lexical_coverage", low_coverage_selected),
):
    for record in selected:
        annotation_rows.append({
            "sample_id": record["sample_id"],
            "sampling_group": group_name,
            "split": record["split"],
            "row_index": record["row_index"],
            "type": record["type"],
            "subtype": record["subtype"],
            "summary_en": record["summary_en"],
            "diff": record["diff"],
            "unmatched_identifiers_heuristic": "; ".join(record["unmatched_identifiers"]),
            "lexical_coverage_heuristic": round(record["lexical_coverage"], 4),
            "summary_terms_not_found_in_diff": "; ".join(record["summary_terms_not_found_in_diff"]),
            "human_label": "",
            "annotator_note": "",
        })

annotation_df = pd.DataFrame(annotation_rows)
assert len(annotation_df) == 100
assert annotation_df["sample_id"].nunique() == 100

output_path = OUTPUT_DIR / "manual_labels_100.csv"
annotation_df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(annotation_df.groupby("sampling_group").size())
print("Annotation CSV:", output_path)
print("Random seed:", RANDOM_SEED)
print("Valid labels:", LABEL_VALUES)

In [ ]:
from pathlib import Path
import hashlib
import json
import os

import pandas as pd
from transformers import AutoTokenizer

# ============================================================
# 1. CAU HINH DUONG DAN
# ============================================================

DATA_DIR = Path("/kaggle/working/CodeFuse123/dataset")

TRAIN_PATH = DATA_DIR / "train.json"
TEST_PATH = DATA_DIR / "test.json"

# Neu da upload tokenizer/model vao Kaggle Input,
# sua MODEL_DIR thanh duong dan thu muc chua tokenizer.
MODEL_DIR = None

MODEL_ID = "meta-llama/Llama-2-7b-hf"
MODEL_REVISION_LOCK = None

MAX_LENGTH = 512

OUTPUT_DIR = Path("/kaggle/working/improve_step1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. KIEM TRA VA DOC DATASET
# ============================================================

if not TRAIN_PATH.is_file():
    raise FileNotFoundError(
        f"Khong tim thay train.json: {TRAIN_PATH}"
    )

if not TEST_PATH.is_file():
    raise FileNotFoundError(
        f"Khong tim thay test.json: {TEST_PATH}"
    )

with TRAIN_PATH.open("r", encoding="utf-8") as f:
    train_rows = json.load(f)

with TEST_PATH.open("r", encoding="utf-8") as f:
    test_rows = json.load(f)

if not isinstance(train_rows, list) or not isinstance(test_rows, list):
    raise ValueError(
        "train.json va test.json phai chua JSON array (list)."
    )

if not train_rows or not test_rows:
    raise ValueError("Train hoac test dang rong.")

required_keys = {"change", "summaries", "type"}

for split_name, rows in [
    ("train", train_rows),
    ("test", test_rows),
]:
    for index, row in enumerate(rows):
        if not isinstance(row, dict):
            raise ValueError(
                f"{split_name}[{index}] khong phai JSON object."
            )

        missing = required_keys - set(row.keys())
        if missing:
            raise ValueError(
                f"{split_name}[{index}] thieu cac truong: {missing}. "
                f"Cac truong hien co: {list(row.keys())}"
            )

        if not isinstance(row["change"], str):
            raise ValueError(
                f"{split_name}[{index}]['change'] phai la chuoi."
            )

        if not isinstance(row["summaries"], dict):
            raise ValueError(
                f"{split_name}[{index}]['summaries'] phai la object."
            )

        if "en" not in row["summaries"]:
            raise ValueError(
                f"{split_name}[{index}]['summaries'] thieu truong 'en'."
            )

print("=" * 60)
print("DATASET")
print("=" * 60)
print(f"Dataset directory: {DATA_DIR}")
print(f"Train file: {TRAIN_PATH}")
print(f"Test file:  {TEST_PATH}")
print(f"So mau train: {len(train_rows):,}")
print(f"So mau test:  {len(test_rows):,}")


# ============================================================
# 3. TAO CHSUM PROMPT
# ============================================================

def make_chsum_prompt(diff):
    return (
        "Please generate a commit message for the following diff:\n\n"
        f"```diff\n{diff}\n```\n"
    )


# ============================================================
# 4. LOAD TOKENIZER
# ============================================================

hf_token = None

if MODEL_DIR is not None:
    model_source = str(Path(MODEL_DIR))

    if not Path(model_source).is_dir():
        raise FileNotFoundError(
            f"MODEL_DIR khong ton tai: {model_source}"
        )

    tokenizer_kwargs = {"local_files_only": True}

else:
    model_source = MODEL_ID

    # Lay token tu Kaggle Secrets hoac environment variable
    try:
        from kaggle_secrets import UserSecretsClient

        hf_token = UserSecretsClient().get_secret("kaggle-llama")

    except Exception:
        hf_token = os.environ.get("kaggle-llama")

    if not hf_token:
        raise RuntimeError(
            "Khong tim thay HF_TOKEN. Hay them HF_TOKEN vao "
            "Kaggle Secrets, hoac dat MODEL_DIR den thu muc "
            "tokenizer da upload vao Kaggle."
        )

    tokenizer_kwargs = {"token": hf_token}

    if MODEL_REVISION_LOCK:
        tokenizer_kwargs["revision"] = MODEL_REVISION_LOCK

tokenizer = AutoTokenizer.from_pretrained(
    model_source,
    **tokenizer_kwargs,
)

resolved_revision = tokenizer.init_kwargs.get("_commit_hash")


# ============================================================
# 5. TINH FINGERPRINT TOKENIZER
# ============================================================

if MODEL_DIR is not None:
    tokenizer_file_names = {
        "tokenizer_config.json",
        "special_tokens_map.json",
        "tokenizer.json",
        "tokenizer.model",
        "vocab.json",
        "merges.txt",
        "added_tokens.json",
    }

    fingerprint = hashlib.sha256()
    local_path = Path(MODEL_DIR)

    for file_path in sorted(
        path
        for path in local_path.rglob("*")
        if path.is_file()
        and path.name in tokenizer_file_names
    ):
        relative_name = file_path.relative_to(
            local_path
        ).as_posix()

        fingerprint.update(relative_name.encode("utf-8"))
        fingerprint.update(file_path.read_bytes())

    tokenizer_fingerprint = fingerprint.hexdigest()
    resolved_revision = resolved_revision or "local-upload"

else:
    tokenizer_fingerprint = None

print("\n" + "=" * 60)
print("TOKENIZER")
print("=" * 60)
print(f"Tokenizer source: {model_source}")
print(f"Resolved revision: {resolved_revision}")
print(f"Max length: {MAX_LENGTH}")


# ============================================================
# 6. THONG KE SO TOKEN CHO TRAIN VA TEST
# ============================================================

summary_rows = []
sample_rows = []

for split_name, rows in [
    ("train", train_rows),
    ("test", test_rows),
]:
    print(f"\nDang xu ly {split_name}: {len(rows):,} mau...")

    for index, row in enumerate(rows):
        diff = row["change"]

        # Token cua diff goc
        diff_token_count = len(
            tokenizer(
                diff,
                add_special_tokens=True,
                truncation=False,
            )["input_ids"]
        )

        # Token cua prompt CHSUM
        chsum_prompt = make_chsum_prompt(diff)

        prompt_token_count = len(
            tokenizer(
                chsum_prompt,
                add_special_tokens=True,
                truncation=False,
            )["input_ids"]
        )

        sample_rows.append({
            "split": split_name,
            "index": index,
            "diff_tokens": diff_token_count,
            "chsum_prompt_tokens": prompt_token_count,
        })

    # Lay cac mau cua split hien tai
    split_samples = [
        item
        for item in sample_rows
        if item["split"] == split_name
    ]

    for input_name, length_key in [
        ("diff_only", "diff_tokens"),
        ("chsum_prompt", "chsum_prompt_tokens"),
    ]:
        lengths = pd.Series(
            [item[length_key] for item in split_samples],
            dtype="int64",
        )

        over_limit = int((lengths > MAX_LENGTH).sum())

        summary_rows.append({
            "split": split_name,
            "input_view": input_name,
            "samples": len(lengths),
            "max_length": MAX_LENGTH,
            "over_limit_count": over_limit,
            "over_limit_rate": (
                over_limit / len(lengths)
                if len(lengths) > 0 else 0.0
            ),
            "mean_tokens": float(lengths.mean()),
            "median_tokens": float(lengths.median()),
            "p90_tokens": float(lengths.quantile(0.90)),
            "max_tokens": int(lengths.max()),
            "model_id": model_source,
            "model_revision": resolved_revision,
        })

summary = pd.DataFrame(summary_rows)
sample_df = pd.DataFrame(sample_rows)


# ============================================================
# 7. LUU KET QUA CSV
# ============================================================

summary_path = OUTPUT_DIR / "prompt_length_summary.csv"
sample_path = OUTPUT_DIR / "prompt_token_lengths.csv"

summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8",
)

sample_df.to_csv(
    sample_path,
    index=False,
    encoding="utf-8",
)


# ============================================================
# 8. LUU METADATA
# ============================================================

metadata = {
    "model_id": model_source,
    "model_revision": resolved_revision,
    "model_revision_lock_used": MODEL_REVISION_LOCK,
    "tokenizer_fingerprint_sha256": tokenizer_fingerprint,
    "dataset_dir": str(DATA_DIR),
    "train_path": str(TRAIN_PATH),
    "test_path": str(TEST_PATH),
    "train_samples": len(train_rows),
    "test_samples": len(test_rows),
    "max_length": MAX_LENGTH,
    "views": {
        "diff_only": (
            "Raw code diff with tokenizer special tokens."
        ),
        "chsum_prompt": (
            "CHSUM wrapper around the raw code diff."
        ),
    },
}

metadata_path = OUTPUT_DIR / "tokenizer_audit_metadata.json"

with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# 9. HIEN THI KET QUA
# ============================================================

print("\n" + "=" * 60)
print("TOKEN LENGTH SUMMARY")
print("=" * 60)

display(summary)

print("\n" + "=" * 60)
print("HOAN TAT")
print("=" * 60)
print(f"Output directory: {OUTPUT_DIR}")
print(f"Summary CSV: {summary_path}")
print(f"Sample CSV:  {sample_path}")
print(f"Metadata:    {metadata_path}")


In [ ]:
import shutil

zip_path = shutil.make_archive(
    "/kaggle/working/improve_step1_results",
    "zip",
    root_dir="/kaggle/working",
    base_dir="improve_step1",
)
print("ZIP kết quả:", zip_path)